![DB Academy](https://files.training.databricks.com/binder/prod_main/get-started-with-databricks-for-data-warehousing-en_us-3.0.3/images/20261012T131742Z/Get Started with Databricks for Data Warehousing/Includes/images/db-academy.png)

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 02</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Querying the Vintage Audio Warehouse</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Discover the lesson tables in Catalog Explorer, write your first SQL queries against governed Unity Catalog tables, and turn the results into a visualization, all from a single notebook.</div>
</div>

</div>

## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

This notebook runs on Serverless compute. Before running any cells, confirm Serverless is selected in the compute dropdown at the top-right of this notebook. This course was developed and tested on <strong>Serverless environment version 5</strong>.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. The setup script will initialize your catalog and schema so your queries run against the correct context.

In [0]:
%run ./Includes/Classroom-Setup-02

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">What a Warehousing Practitioner Does</div>
<div style="font-size: 13pt; color: #5A6F77; margin-bottom: 24px;">Day-to-day warehousing work breaks into three phases. Each one is traditionally slowed down by data living in too many places and tools being scattered across the stack. Databricks brings data, compute, modeling, and BI delivery into a single governed environment.</div>

<div style="display: flex; justify-content: center; align-items: stretch; gap: 24px; flex-wrap: wrap;">

  <div style="width: 300px; min-height: 220px; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); display: flex; flex-direction: column; justify-content: center; gap: 10px; padding: 24px; text-align: center; position: relative; box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 8px; background: #4299E0; border-radius: 8px 8px 0 0;"></div>
    <div style="font-size: 16pt; font-weight: 600; color: #0b2026;">Discover &amp; Load</div>
    <div style="font-size: 13pt; color: #5E7077;">Find or land the data the business depends on, often arriving as files or pulled from operational systems.</div>
  </div>

  <div style="width: 300px; min-height: 220px; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); display: flex; flex-direction: column; justify-content: center; gap: 10px; padding: 24px; text-align: center; position: relative; box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 8px; background: #FE3722; border-radius: 8px 8px 0 0;"></div>
    <div style="font-size: 16pt; font-weight: 600; color: #0b2026;">Model &amp; Tune</div>
    <div style="font-size: 13pt; color: #5E7077;">Shape the data into a model the business can reason about (typically dimensions and facts), and tune for fast analytical reads.</div>
  </div>

  <div style="width: 300px; min-height: 220px; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); display: flex; flex-direction: column; justify-content: center; gap: 10px; padding: 24px; text-align: center; position: relative; box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 8px; background: #00A972; border-radius: 8px 8px 0 0;"></div>
    <div style="font-size: 16pt; font-weight: 600; color: #0b2026;">Deliver &amp; Tune</div>
    <div style="font-size: 13pt; color: #5E7077;">Get the right metrics to stakeholders through dashboards, queries, and BI tools, and keep them fast as data grows.</div>
  </div>

</div>



</div>

## 1. Where data lives: the three-level namespace

Every table in Databricks lives at a **three-level address**: `catalog.schema.table`. Catalogs are the top-level container, schemas group related tables inside a catalog, and tables hold the data itself. Volumes (used for files) also live inside a schema. The diagram below shows the layout you have in your workspace right now, after running the setup script.

<div style="width: 100%; max-width: 920px; margin: 0 auto;">
<svg viewBox="0 0 920 436" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
  <text x="460" y="28" text-anchor="middle" fill="#1B5162" font-size="20" font-weight="700" letter-spacing="1.5">THE THREE-LEVEL NAMESPACE</text>
  <!-- Metastore container -->
  <rect x="16" y="44" width="888" height="240" rx="10" fill="#F9F7F4" stroke="#1B5162" stroke-width="1.5"/>
  <text x="36" y="70" fill="#1B5162" font-size="16" font-weight="700" letter-spacing="1.5">METASTORE</text>
  <!-- Catalog container -->
  <rect x="36" y="82" width="848" height="190" rx="8" fill="#FFFFFF" stroke="#FE3722" stroke-width="2"/>
  <!-- Catalog info (left of schema) -->
  <text x="56" y="140" fill="#FE3722" font-size="16" font-weight="700" letter-spacing="1.5">CATALOG</text>
  <text x="56" y="168" fill="#0b2026" font-size="18" font-weight="700">labuser_yourname</text>
  <text x="56" y="192" fill="#5E7077" font-size="16" font-style="italic">your personal</text>
  <text x="56" y="212" fill="#5E7077" font-size="16" font-style="italic">catalog</text>
  <!-- Schema box (right of catalog info) -->
  <rect x="230" y="98" width="636" height="160" rx="8" fill="#F9F7F4" stroke="#FE3722" stroke-width="2"/>
  <text x="250" y="124" fill="#FE3722" font-size="16" font-weight="700" letter-spacing="1.5">SCHEMA</text>
  <text x="250" y="148" fill="#0b2026" font-size="18" font-weight="700">vintage_audio</text>
  <!-- Tables -->
  <rect x="250" y="162" width="192" height="48" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.2"/>
  <text x="346" y="183" text-anchor="middle" fill="#0b2026" font-size="16" font-weight="700">customers</text>
  <text x="346" y="201" text-anchor="middle" fill="#5E7077" font-size="16">TABLE</text>
  <rect x="452" y="162" width="192" height="48" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.2"/>
  <text x="548" y="183" text-anchor="middle" fill="#0b2026" font-size="16" font-weight="700">products</text>
  <text x="548" y="201" text-anchor="middle" fill="#5E7077" font-size="16">TABLE</text>
  <rect x="654" y="162" width="192" height="48" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.2"/>
  <text x="750" y="183" text-anchor="middle" fill="#0b2026" font-size="16" font-weight="700">stores</text>
  <text x="750" y="201" text-anchor="middle" fill="#5E7077" font-size="16">TABLE</text>
  <!-- Volume -->
  <rect x="250" y="220" width="596" height="24" rx="5" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.2" stroke-dasharray="4,3"/>
  <text x="548" y="237" text-anchor="middle" fill="#5E7077" font-size="16"><tspan font-weight="700" fill="#0b2026">data</tspan>  &#8212;  VOLUME (CSV files staged for Lesson 06)</text>
  <!-- Full table address callout -->
  <rect x="16" y="308" width="888" height="120" rx="10" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
  <text x="36" y="334" fill="#1B5162" font-size="16" font-weight="700" letter-spacing="1.5">FULL TABLE ADDRESS</text>
  <!-- Color-coded address segments -->
  <rect x="34" y="346" width="167" height="24" rx="3" fill="#FFF0EF"/>
  <rect x="211" y="346" width="137" height="24" rx="3" fill="#FFF5ED"/>
  <rect x="358" y="346" width="94" height="24" rx="3" fill="#EDF5F7"/>
  <text x="36" y="364" font-size="17" font-family="monospace" font-weight="700"><tspan fill="#FE3722">labuser_yourname</tspan><tspan fill="#5E7077">.</tspan><tspan fill="#FF8C42">vintage_audio</tspan><tspan fill="#5E7077">.</tspan><tspan fill="#1B5162">customers</tspan></text>
  <text x="36" y="388" fill="#FE3722" font-size="16" font-family="monospace">&#8593; catalog</text>
  <text x="206" y="388" fill="#FF8C42" font-size="16" font-family="monospace">&#8593; schema</text>
  <text x="351" y="388" fill="#1B5162" font-size="16" font-family="monospace">&#8593; table</text>
  <text x="36" y="416" fill="#5E7077" font-size="16" font-style="italic">After setup runs, USE CATALOG and USE SCHEMA are set, so you can refer to tables by short name (e.g. customers).</text>
</svg>
</div>

### Confirm where you are

The cell below confirms what catalog and schema your session is pointing at right now.

In [0]:
%sql
SELECT current_catalog(), current_schema();

You should see your personal catalog (something like `labuser_yourname`) and the `vintage_audio` schema. Every query in this notebook will run against that schema unless you say otherwise.

## 2. List the tables in this schema

The `SHOW TABLES` command lists every table in the current schema. Run it to see what was created for you by the setup script.

In [0]:
%sql
SHOW TABLES;

You should see three tables: **`customers`**, **`products`**, and **`stores`**. These are the three dimension sources for Vintage Audio Co.

The order data has not been loaded into a table yet. Its CSV files are sitting in the `data` volume, waiting for Lesson 06 to load them.

## 3. Inspect a table with DESCRIBE EXTENDED

Before querying a table for the first time, it's useful to see its full schema and where the data physically lives. `DESCRIBE EXTENDED` returns columns, data types, storage location, format, and other table metadata.

<div style="max-width: 880px; margin: 20px auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="font-size: 15px; color: #5E7077; margin-bottom: 14px;">The output has three sections separated by blank rows:</div>
  <div style="display: flex; gap: 14px; align-items: stretch;">
    <!-- Part 1: Columns -->
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 26px; height: 26px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px;">1</span>
        Column Schema
      </div>
      <div style="padding: 14px; background: #f8fafc; flex: 1;">
        <div style="font-size: 15px; color: #3d5a65; margin-bottom: 10px;"><strong>Names + data types</strong></div>
        <div style="background: #fff; border: 1px solid #DCE0E2; border-radius: 6px; padding: 10px; font-family: monospace; font-size: 14px; line-height: 1.7; color: #1B3139;">
          customer_id&nbsp;&nbsp;&nbsp;&nbsp;string<br/>
          customer_name&nbsp;&nbsp;string<br/>
          city&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;string<br/>
          postal_code&nbsp;&nbsp;&nbsp;&nbsp;int<br/>
          <span style="color: #5E7077;">... 6 more columns</span>
        </div>
      </div>
    </div>
    <!-- Part 2: Delta Statistics -->
    <div style="flex: 1; border: 2px solid #FF8C42; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #FF8C42; color: #fff; padding: 10px 14px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 26px; height: 26px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px;">2</span>
        Delta Statistics
      </div>
      <div style="padding: 14px; background: #FFFAF6; flex: 1;">
        <div style="font-size: 15px; color: #3d5a65; margin-bottom: 10px;"><strong>Min/max stats for file skipping</strong></div>
        <div style="display: flex; flex-direction: column; gap: 6px;">
          <div style="background: #fff; border: 1px solid #E8D5C4; border-radius: 6px; padding: 8px 10px; font-size: 15px; display: flex; justify-content: space-between;">
            <span style="color: #5E7077;">Method</span><span style="font-weight: 700; color: #1B3139; font-family: monospace; font-size: 14px;">first-32</span>
          </div>
          <div style="background: #fff; border: 1px solid #E8D5C4; border-radius: 6px; padding: 8px 10px; font-size: 14px; color: #3d5a65; line-height: 1.5;">
            <strong>Tracked columns</strong><br/>
            <span style="font-family: monospace;">customer_id, customer_name, email, city, state, postal_code, ...</span>
          </div>
        </div>
      </div>
    </div>
    <!-- Part 3: Table Info -->
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 10px 14px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 26px; height: 26px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px;">3</span>
        Table Metadata
      </div>
      <div style="padding: 14px; background: #f8fafc; flex: 1;">
        <div style="font-size: 15px; color: #3d5a65; margin-bottom: 10px;"><strong>Location, format, size</strong></div>
        <div style="display: flex; flex-direction: column; gap: 6px;">
          <div style="background: #fff; border: 1px solid #DCE0E2; border-radius: 6px; padding: 8px 10px; font-size: 15px; display: flex; justify-content: space-between;">
            <span style="color: #5E7077;">Provider</span><span style="font-weight: 700; color: #1B3139; font-family: monospace; font-size: 14px;">delta</span>
          </div>
          <div style="background: #fff; border: 1px solid #DCE0E2; border-radius: 6px; padding: 8px 10px; font-size: 15px; display: flex; justify-content: space-between;">
            <span style="color: #5E7077;">Type</span><span style="font-weight: 700; color: #1B3139; font-family: monospace; font-size: 14px;">MANAGED</span>
          </div>
          <div style="background: #fff; border: 1px solid #DCE0E2; border-radius: 6px; padding: 8px 10px; font-size: 15px; display: flex; justify-content: space-between;">
            <span style="color: #5E7077;">Location</span><span style="font-weight: 700; color: #1B3139; font-family: monospace; font-size: 14px;">s3://...</span>
          </div>
          <div style="background: #fff; border: 1px solid #DCE0E2; border-radius: 6px; padding: 8px 10px; font-size: 15px; display: flex; justify-content: space-between;">
            <span style="color: #5E7077;">Statistics</span><span style="font-weight: 700; color: #1B3139; font-family: monospace; font-size: 14px;">5,000 rows</span>
          </div>
        </div>
      </div>
    </div>
  </div>
</div>

In [0]:
%sql
DESCRIBE EXTENDED customers;

Reading the result from top to bottom, it has three parts:

1. **Column names and data types** : one row per column (`customer_id`, `customer_name`, `email`, `address`, `city`, `state`, and so on), each with its data type.
2. **Delta statistics columns** : a short block listing the columns Delta collects statistics on, which it uses to skip data it doesn't need to read.
3. **Detailed table information** : table-level metadata, including the storage `Location` (a path under your catalog's managed storage) and the `Provider` (which will be `delta`).

A blank separator row sits between each part. None of the course tables are partitioned, so the `Partition Information` is empty. (Partitioning is one performance lever, but not always the best one. Lesson 04 covers a more effective lever for analytical workloads: liquid clustering.)

## 4. Your first query: select a few rows

`SELECT * FROM <table> LIMIT 10` is a quick way to see what's actually in a table. Run it for `customers`.

In [0]:
%sql
SELECT * FROM customers LIMIT 10;

You should see 10 rows. Each customer has a `customer_id`, a name, an email and phone, a US address, a `customer_since_date`, and a `loyalty_tier` (one of `Bronze`, `Silver`, `Gold`, `Platinum`).

## 5. Filter and aggregate: customers by state

Analytical SQL is usually about **aggregating** (counting, summing, averaging) **grouped by** some dimension. The query below groups customers by their home state and counts them.

<div style="width: 100%; margin: 20px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 16px;">
  <div style="display: flex; gap: 16px; align-items: center;">
    <!-- Step 1: Raw rows -->
    <div style="flex: 0 0 auto;">
      <div style="font-size: 16px; font-weight: 700; color: #1B5162; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 8px; text-align: center;">customers table</div>
      <div style="background: #fff; border: 1.5px solid #DCE0E2; border-radius: 8px; overflow: hidden;">
        <div style="display: grid; grid-template-columns: auto 1fr; font-size: 16px;">
          <div style="padding: 8px 14px; font-weight: 700; color: #5E7077; background: #f4f6f7; border-bottom: 1px solid #DCE0E2; border-right: 1px solid #DCE0E2;">name</div>
          <div style="padding: 8px 14px; font-weight: 700; color: #5E7077; background: #f4f6f7; border-bottom: 1px solid #DCE0E2;">state</div>
          <div style="padding: 7px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; color: #3d5a65;">Alice</div>
          <div style="padding: 7px 14px; border-bottom: 1px solid #eee; font-weight: 600; color: #1B5162; background: #EDF5F7;">CA</div>
          <div style="padding: 7px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; color: #3d5a65;">Bob</div>
          <div style="padding: 7px 14px; border-bottom: 1px solid #eee; font-weight: 600; color: #FF8C42; background: #FFF5ED;">TX</div>
          <div style="padding: 7px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; color: #3d5a65;">Carol</div>
          <div style="padding: 7px 14px; border-bottom: 1px solid #eee; font-weight: 600; color: #1B5162; background: #EDF5F7;">CA</div>
          <div style="padding: 7px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; color: #3d5a65;">Dan</div>
          <div style="padding: 7px 14px; border-bottom: 1px solid #eee; font-weight: 600; color: #1B5162; background: #EDF5F7;">CA</div>
          <div style="padding: 7px 14px; border-right: 1px solid #eee; color: #3d5a65;">Eve</div>
          <div style="padding: 7px 14px; font-weight: 600; color: #FF8C42; background: #FFF5ED;">TX</div>
        </div>
      </div>
      <div style="text-align: center; font-size: 15px; color: #5E7077; margin-top: 6px;">5,000 rows</div>
    </div>
    <!-- Arrow 1 -->
    <div style="flex: 0 0 auto; text-align: center; padding: 0 8px;">
      <div style="font-size: 16px; color: #5E7077; font-weight: 600; margin-bottom: 4px;">GROUP BY<br/>state</div>
      <div style="font-size: 28px; color: #1B5162;">&#x2192;</div>
    </div>
    <!-- Step 2: Grouped -->
    <div style="flex: 0 0 auto;">
      <div style="font-size: 16px; font-weight: 700; color: #1B5162; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 8px; text-align: center;">Grouped buckets</div>
      <div style="display: flex; flex-direction: column; gap: 10px;">
        <div style="background: #EDF5F7; border: 1.5px solid #1B5162; border-radius: 8px; padding: 12px 20px; text-align: center;">
          <div style="font-size: 17px; font-weight: 700; color: #1B5162;">CA</div>
          <div style="font-size: 15px; color: #5E7077;">Alice, Carol, Dan</div>
        </div>
        <div style="background: #FFF5ED; border: 1.5px solid #FF8C42; border-radius: 8px; padding: 12px 20px; text-align: center;">
          <div style="font-size: 17px; font-weight: 700; color: #FF8C42;">TX</div>
          <div style="font-size: 15px; color: #5E7077;">Bob, Eve</div>
        </div>
        <div style="background: #f4f6f7; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 10px 20px; text-align: center;">
          <div style="font-size: 15px; color: #5E7077;">... more states</div>
        </div>
      </div>
    </div>
    <!-- Arrow 2 -->
    <div style="flex: 0 0 auto; text-align: center; padding: 0 8px;">
      <div style="font-size: 16px; color: #5E7077; font-weight: 600; margin-bottom: 4px;">COUNT(*)</div>
      <div style="font-size: 28px; color: #1B5162;">&#x2192;</div>
    </div>
    <!-- Step 3: Result -->
    <div style="flex: 1;">
      <div style="font-size: 16px; font-weight: 700; color: #1B5162; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 8px; text-align: center;">Result</div>
      <div style="background: #fff; border: 1.5px solid #DCE0E2; border-radius: 8px; overflow: hidden;">
        <div style="display: grid; grid-template-columns: 1fr 1fr; font-size: 16px;">
          <div style="padding: 8px 14px; font-weight: 700; color: #5E7077; background: #f4f6f7; border-bottom: 1px solid #DCE0E2; border-right: 1px solid #DCE0E2;">state</div>
          <div style="padding: 8px 14px; font-weight: 700; color: #5E7077; background: #f4f6f7; border-bottom: 1px solid #DCE0E2;">customer_count</div>
          <div style="padding: 7px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; font-weight: 600; color: #1B5162;">CA</div>
          <div style="padding: 7px 14px; border-bottom: 1px solid #eee; color: #1B3139; font-weight: 700;">3</div>
          <div style="padding: 7px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; font-weight: 600; color: #FF8C42;">TX</div>
          <div style="padding: 7px 14px; border-bottom: 1px solid #eee; color: #1B3139; font-weight: 700;">2</div>
          <div style="padding: 7px 14px; border-right: 1px solid #eee; color: #5E7077;">...</div>
          <div style="padding: 7px 14px; color: #5E7077;">...</div>
        </div>
      </div>
      <div style="text-align: center; font-size: 14px; color: #5E7077; margin-top: 6px;">One row per state, sorted DESC, top 15</div>
    </div>
  </div>
</div>

In [0]:
%sql
SELECT
  state,
  COUNT(*) AS customer_count
FROM customers
GROUP BY state
ORDER BY customer_count DESC
LIMIT 15;

The result is one row per state, sorted by `customer_count` descending. The top states tend to be the populous ones (California, Texas, Florida), which is what you would expect from a US retailer.

## 6. Combine two tables with a join

So far every query has read from a single table. A **join** lets one query pull columns from two tables at once. Most joins match rows using a shared key, for example pairing each order with the customer who placed it. You will work with those key-based joins once the `orders` fact table exists in Lesson 08.

There is one join that needs no matching key: a **cross join**. A cross join pairs *every* row in the first table with *every* row in the second, so the result has one row for each possible combination. The diagram below shows the idea with a tiny example.

<div style="width: 100%; margin: 0 auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="text-align: center; font-size: 18px; font-weight: 700; color: #1B5162; margin-bottom: 16px;">CROSS JOIN: every row paired with every row</div>
  <div style="display: flex; gap: 16px; align-items: flex-start;">
    <!-- Left: source tables -->
    <div style="display: flex; gap: 10px; align-items: flex-start; flex: 0 0 auto;">
      <!-- Tiers column -->
      <div>
        <div style="font-size: 15px; font-weight: 700; color: #FE3722; text-align: center; margin-bottom: 6px;">tiers</div>
        <div style="background: #fff; border: 2px solid #FE3722; border-radius: 8px; padding: 12px 20px; text-align: center; font-size: 16px; color: #0b2026; margin-bottom: 6px;">Gold</div>
        <div style="background: #fff; border: 2px solid #FE3722; border-radius: 8px; padding: 12px 20px; text-align: center; font-size: 16px; color: #0b2026;">Platinum</div>
      </div>
      <!-- X symbol -->
      <div style="padding-top: 36px; font-size: 22px; font-weight: 700; color: #1B5162;">&#215;</div>
      <!-- Categories column -->
      <div>
        <div style="font-size: 15px; font-weight: 700; color: #00A972; text-align: center; margin-bottom: 6px;">categories</div>
        <div style="background: #fff; border: 2px solid #00A972; border-radius: 8px; padding: 12px 20px; text-align: center; font-size: 16px; color: #0b2026; margin-bottom: 6px;">Speakers</div>
        <div style="background: #fff; border: 2px solid #00A972; border-radius: 8px; padding: 12px 20px; text-align: center; font-size: 16px; color: #0b2026; margin-bottom: 6px;">Amplifiers</div>
        <div style="background: #fff; border: 2px solid #00A972; border-radius: 8px; padding: 12px 20px; text-align: center; font-size: 16px; color: #0b2026;">Turntables</div>
      </div>
    </div>
    <!-- Arrow -->
    <div style="padding-top: 60px; font-size: 28px; color: #1B5162;">&#8594;</div>
    <!-- Right: result as a proper table -->
    <div style="flex: 1;">
      <div style="font-size: 15px; font-weight: 700; color: #1B5162; margin-bottom: 6px;">result: 2 &#215; 3 = 6 rows (with actual counts from the query)</div>
      <div style="background: #fff; border: 2px solid #1B5162; border-radius: 8px; overflow: hidden;">
        <div style="display: grid; grid-template-columns: auto auto auto; font-size: 15px;">
          <div style="padding: 8px 14px; font-weight: 700; color: #5E7077; background: #f4f6f7; border-bottom: 2px solid #DCE0E2; border-right: 1px solid #DCE0E2;">loyalty_tier</div>
          <div style="padding: 8px 14px; font-weight: 700; color: #5E7077; background: #f4f6f7; border-bottom: 2px solid #DCE0E2; border-right: 1px solid #DCE0E2;">category</div>
          <div style="padding: 8px 14px; font-weight: 700; color: #5E7077; background: #f4f6f7; border-bottom: 2px solid #DCE0E2; text-align: right;">row_count</div>
          <div style="padding: 8px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; font-weight: 600; color: #FE3722; background: #FFF8F7;">Gold</div>
          <div style="padding: 8px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; color: #1B3139; background: #FFF8F7;">Speakers</div>
          <div style="padding: 8px 14px; border-bottom: 1px solid #eee; color: #1B3139; background: #FFF8F7; text-align: right; font-weight: 600;">21,455</div>
          <div style="padding: 8px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; font-weight: 600; color: #FE3722; background: #FFF8F7;">Gold</div>
          <div style="padding: 8px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; color: #1B3139; background: #FFF8F7;">Amplifiers</div>
          <div style="padding: 8px 14px; border-bottom: 1px solid #eee; color: #1B3139; background: #FFF8F7; text-align: right; font-weight: 600;">24,520</div>
          <div style="padding: 8px 14px; border-bottom: 2px solid #DCE0E2; border-right: 1px solid #eee; font-weight: 600; color: #FE3722; background: #FFF8F7;">Gold</div>
          <div style="padding: 8px 14px; border-bottom: 2px solid #DCE0E2; border-right: 1px solid #eee; color: #1B3139; background: #FFF8F7;">Turntables</div>
          <div style="padding: 8px 14px; border-bottom: 2px solid #DCE0E2; color: #1B3139; background: #FFF8F7; text-align: right; font-weight: 600;">36,780</div>
          <div style="padding: 8px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; font-weight: 600; color: #FE3722;">Platinum</div>
          <div style="padding: 8px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; color: #1B3139;">Speakers</div>
          <div style="padding: 8px 14px; border-bottom: 1px solid #eee; color: #1B3139; text-align: right; font-weight: 600;">7,175</div>
          <div style="padding: 8px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; font-weight: 600; color: #FE3722;">Platinum</div>
          <div style="padding: 8px 14px; border-bottom: 1px solid #eee; border-right: 1px solid #eee; color: #1B3139;">Amplifiers</div>
          <div style="padding: 8px 14px; border-bottom: 1px solid #eee; color: #1B3139; text-align: right; font-weight: 600;">8,200</div>
          <div style="padding: 8px 14px; border-right: 1px solid #eee; font-weight: 600; color: #FE3722;">Platinum</div>
          <div style="padding: 8px 14px; border-right: 1px solid #eee; color: #1B3139;">Turntables</div>
          <div style="padding: 8px 14px; color: #1B3139; text-align: right; font-weight: 600;">12,300</div>
        </div>
      </div>
    </div>
  </div>
</div>

### Build this query with Genie Code (AI-assisted)

Instead of typing the SQL by hand, describe what you want in plain language and let **Genie Code** write it. Genie Code is the natural-language code assistant built into the notebook.

<div style="width: 100%; margin: 20px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 16px;">
  <!-- 3-step flow -->
  <div style="display: flex; gap: 0; align-items: stretch;">
    <!-- Step 1: How to open -->
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 12px 16px; font-size: 17px; font-weight: 700; display: flex; align-items: center; gap: 10px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 28px; height: 28px; display: inline-flex; align-items: center; justify-content: center; font-size: 16px; flex-shrink: 0;">1</span>
        Open Genie Code
      </div>
      <div style="padding: 16px; background: #f8fafc; flex: 1; color: #3d5a65; line-height: 1.6;">
        <div style="margin-bottom: 10px;"><strong>Inline:</strong> <code style="font-size: 16px; background: #EEEDE9; padding: 2px 8px; border-radius: 3px;">Cmd/Ctrl + I</code> inside a code cell</div>
        <div><strong>Sidebar:</strong> Click the <strong>Genie / Assistant</strong> icon in the right panel</div>
      </div>
    </div>
    <!-- Arrow -->
    <div style="flex: 0 0 44px; display: flex; align-items: center; justify-content: center; font-size: 26px; color: #1B5162;">&#x2192;</div>
    <!-- Step 2: Describe -->
    <div style="flex: 1.6; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 12px 16px; font-size: 17px; font-weight: 700; display: flex; align-items: center; gap: 10px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 28px; height: 28px; display: inline-flex; align-items: center; justify-content: center; font-size: 16px; flex-shrink: 0;">2</span>
        Paste this prompt
      </div>
      <div style="padding: 16px; background: #f8fafc; flex: 1;">
        <div id="genie-prompt-text" style="background: #fff; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 14px 16px; color: #3d5a65; line-height: 1.5; font-style: italic;">For Gold and Platinum customers only, pair every loyalty tier with every product category and count the combinations. Return loyalty_tier, category, and the count aliased as row_count, sorted by tier then count descending. Use the customers and products tables.</div>
        <button onclick="var t=document.getElementById('genie-prompt-text').textContent,a=document.createElement('textarea');a.value=t;a.style.position='fixed';a.style.left='-9999px';document.body.appendChild(a);a.select();document.execCommand('copy');document.body.removeChild(a);this.textContent='Copied!';this.style.background='#00A972';this.style.borderColor='#00A972';var b=this;setTimeout(function(){b.textContent='Copy prompt';b.style.background='#1B5162';b.style.borderColor='#1B5162'},1500)" style="margin-top: 12px; background: #1B5162; color: #fff; border: 2px solid #1B5162; border-radius: 6px; padding: 8px 20px; font-size: 16px; font-weight: 600; cursor: pointer;">Copy prompt</button>
      </div>
    </div>
    <!-- Arrow -->
    <div style="flex: 0 0 44px; display: flex; align-items: center; justify-content: center; font-size: 26px; color: #1B5162;">&#x2192;</div>
    <!-- Step 3: Review + Run -->
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 12px 16px; font-size: 17px; font-weight: 700; display: flex; align-items: center; gap: 10px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 28px; height: 28px; display: inline-flex; align-items: center; justify-content: center; font-size: 16px; flex-shrink: 0;">3</span>
        Review and run
      </div>
      <div style="padding: 16px; background: #f8fafc; flex: 1; color: #3d5a65; line-height: 1.6;">
        <div style="margin-bottom: 10px;">Genie Code generates SQL with <code style="font-size: 16px; background: #EEEDE9; padding: 2px 8px; border-radius: 3px;">CROSS JOIN</code></div>
        <div>Review the output, run it, and compare against the reference below</div>
      </div>
    </div>
  </div>
</div>

Make sure the empty code cell below is the target, then paste the prompt from Step 2 into Genie Code.

In [0]:
%sql
-- Run the query Genie Code generated here.
-- If you'd rather check your result, the reference query is in the expandable section below.

<details>
<summary><strong>Here is the full cross-join query</strong></summary>
<pre><code>SELECT
  c.loyalty_tier,
  p.category,
  COUNT(*) AS row_count
FROM customers c
CROSS JOIN products p
WHERE c.loyalty_tier IN ('Gold', 'Platinum')
GROUP BY c.loyalty_tier, p.category
ORDER BY c.loyalty_tier, row_count DESC;</code></pre>
</details>

This result is artificial: pairing every tier with every category isn't a business question anyone would ask. It exists to show the cross-join mechanic. In real warehouse work, the common pattern is joining a **fact table** to its dimensions on a matching key, which you'll build starting in Lesson 08.

## 7. Turn a result into a visualization, inline

Databricks notebook results can be turned into a chart without leaving the notebook or writing any visualization code. Run the query below, then turn the result into a bar chart.

<div style="width: 100%; margin: 20px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="display: flex; gap: 0; align-items: stretch;">
    <!-- Step 1 -->
    <div style="flex: 1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 8px 12px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 6px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 24px; height: 24px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px; flex-shrink: 0;">1</span> Run query
      </div>
      <div style="padding: 10px 12px; background: #f8fafc; flex: 1; font-size: 15px; color: #3d5a65; line-height: 1.5;">
        Returns <strong>state</strong> + <strong>customer_count</strong> for the top 10
      </div>
    </div>
    <div style="flex: 0 0 30px; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #1B5162;">&#x2192;</div>
    <!-- Step 2 -->
    <div style="flex: 1.1; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 8px 12px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 6px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 24px; height: 24px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px; flex-shrink: 0;">2</span> Click +
      </div>
      <div style="padding: 10px 12px; background: #f8fafc; flex: 1; font-size: 15px; color: #3d5a65; line-height: 1.5;">
        Above the result table (next to the <strong>Table</strong> tab), click <strong>+</strong> and choose <strong>Visualization</strong>
      </div>
    </div>
    <div style="flex: 0 0 30px; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #1B5162;">&#x2192;</div>
    <!-- Step 3 -->
    <div style="flex: 1.8; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #1B5162; color: #fff; padding: 8px 12px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 6px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 24px; height: 24px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px; flex-shrink: 0;">3</span> Visualization Editor settings
      </div>
      <div style="padding: 10px 12px; background: #f8fafc; flex: 1; font-size: 15px; color: #3d5a65;">
        <div style="display: grid; grid-template-columns: auto auto; gap: 3px 12px; align-items: center;">
          <span style="font-weight: 700; color: #1B5162;">Type:</span><code style="font-size: 14px; background: #EEEDE9; padding: 1px 6px; border-radius: 3px;">Bar</code>
          <span style="font-weight: 700; color: #1B5162;">Horizontal chart:</span><span>&#9745; on</span>
          <span style="font-weight: 700; color: #1B5162;">Y column:</span><code style="font-size: 14px; background: #EEEDE9; padding: 1px 6px; border-radius: 3px;">state</code>
          <span style="font-weight: 700; color: #1B5162;">X columns:</span><code style="font-size: 14px; background: #EEEDE9; padding: 1px 6px; border-radius: 3px;">customer_count</code>
        </div>
      </div>
    </div>
    <div style="flex: 0 0 30px; display: flex; align-items: center; justify-content: center; font-size: 22px; color: #1B5162;">&#x2192;</div>
    <!-- Step 4 -->
    <div style="flex: 0.8; border: 2px solid #00A972; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column;">
      <div style="background: #00A972; color: #fff; padding: 8px 12px; font-size: 15px; font-weight: 700; display: flex; align-items: center; gap: 6px;">
        <span style="background: rgba(255,255,255,0.2); border-radius: 50%; width: 24px; height: 24px; display: inline-flex; align-items: center; justify-content: center; font-size: 14px; flex-shrink: 0;">4</span> Save
      </div>
      <div style="padding: 10px 12px; background: #F0FAF5; flex: 1; font-size: 15px; color: #3d5a65; line-height: 1.5;">
        Chart embeds as a new tab, auto-refreshes
      </div>
    </div>
  </div>
</div>

In [0]:
%sql
SELECT
  state,
  COUNT(*) AS customer_count
FROM customers
GROUP BY state
ORDER BY customer_count DESC
LIMIT 10;

#### Make a bar chart from this result

1. Above the result table, click the **`+`** icon and choose **Visualization**.
2. Set the **Visualization Type** to **`Bar`**.
3. Assign the columns so that **`state`** labels the bars and **`customer_count`** sets their length:
   - If the editor shows a **Horizontal chart** toggle that is **on** by default, the axes are swapped: put **`customer_count`** on the **X column** and **`state`** on the **Y column**.
   - If **Horizontal chart** is **off** (vertical bars), put **`state`** on the **X column** and **`customer_count`** on the **Y column**.

   Either way, you want each bar labeled by a state and sized by its customer count. If the chart looks wrong, the two columns are simply assigned to the wrong axes; swap them.
4. Click **Save**.

You now have a chart embedded in the notebook. It will refresh automatically if the underlying query result changes.

## What you did in this lesson

<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<div style="color:#333;">

In this lesson you:

- Located the course tables inside Unity Catalog using the three-level namespace.
- Ran `SHOW TABLES` and `DESCRIBE EXTENDED` to discover and inspect tables.
- Wrote `SELECT` queries with `WHERE`, `GROUP BY`, and `ORDER BY`.
- Learned what a join does and used Genie Code to build a `CROSS JOIN` query from a plain-language prompt.
- Turned a result set into an inline bar-chart visualization.


</div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>